# Robustness of Accessibility
## Notebook 1/4 - extract and persist data via overpass api using osmnx

In [ ]:
from __future__ import annotations
import osmnx as ox
from shapely.ops import unary_union
import geopandas as gpd
from pathlib import Path
from css_geodata_service.robustness_of_accessibility.examples.notebooks.notebook_utils import (
    get_roa_hazard_data_path,
    get_roa_inputs_path,
    set_notbook_wd,
)


# ox.config(use_cache=True, log_console=True)
ox.settings.log_console = False
ox.settings.use_cache = True
ox.__version__
# set wd using global utils
set_notbook_wd()

### Specify analysis paramter globally in the static class RoaNotebookConfic

In [ ]:
from css_geodata_service.robustness_of_accessibility.examples.notebooks.notebook_utils import RoaNotebookConfig


event = RoaNotebookConfig.event

In [ ]:
from css_geodata_service.robustness_of_accessibility.examples.notebooks.notebook_utils import get_roa_cache_path


inputs_dir: Path = get_roa_inputs_path()
cache_dir: Path = get_roa_cache_path()
hazard_data_path: Path = get_roa_hazard_data_path(event=event)

# Get the bounds of the region from osm
- In order to generate somewhat realistic data we need a road network that expands beyond the area from which samples are drawn
  - Generate a region and add buffer for road network (must still be included within the flooding data)
  - Generate a region a remove a buffer for the sampling algorithm (later)

### Use one way roads?
- determine way of transport: if undirected routes can go in the wrong direction of one way roads e.g. like emergency services might need to do

In [ ]:
undirected_graph: bool = RoaNotebookConfig.undirected_graph

### Select network_type
- determine netowork type i.e. weather to include non public roads etc. select one of {"all", "all_public", "bike", "drive", "drive_service", "walk"}

In [ ]:
network_type: str = RoaNotebookConfig.network_type

### Select place
- Select the place by providing the name of the region or city c.f osmnx docs

In [ ]:
# use name to get bounds
place_name = RoaNotebookConfig.place_name
place_gdf = ox.geocode_to_gdf(place_name)

### Determine the bounds and polygon of that place

In [ ]:
# 1. From the GeoDataFrame
if len(place_gdf) > 1:
    raise RuntimeError(f"gdf for {place_name} has multiple entries, could not safely deterine bounds")
boundary_geom = place_gdf.geometry.iloc[0]  # usually the first geometry

# 2. If it’s a MultiPolygon, merge the parts into one

if boundary_geom.geom_type == "MultiPolygon":
    boundary_geom = unary_union(boundary_geom)
elif boundary_geom.geom_type != "Polygon":
    raise RuntimeError(f"Could not combine boundary due to unexpected data structure {boundary_geom.geom_type}")
print(boundary_geom.bounds)
boundary_geom

In [ ]:
import json

# persist boundary geom
with open(cache_dir / f"services/boundary_geom_{place_name}.geojson", "w") as f:
    json.dump(boundary_geom.__geo_interface__, f, indent=2)

### Derive a the road network
- Since we want to make sure that targets that are on the edge of the investigated area i.e. boundary_geom can actually use roads in their environment event if they are not directly part of the area itself
- Therefore we add a buffer (e.g of 3km) to the original boundary before deriving the road network

In [ ]:
from css_geodata_service.generate.buffer_polygon import add_buffer_in_meters

geom_buffered = add_buffer_in_meters(boundary_geom, 3000)

In [ ]:
import matplotlib.pyplot as plt

# visualize both boundaries for verifcation
# Convert to GeoSeries for plotting
boundary_test_gdf = gpd.GeoSeries([geom_buffered, boundary_geom])

# Plot with different colors and transparency to see overlap
fig, ax = plt.subplots()
boundary_test_gdf.plot(ax=ax, color=["skyblue", "salmon"], alpha=0.6, edgecolor="black")

# Equal aspect ratio
ax.set_aspect("equal")
plt.show()

In [ ]:
road_network = ox.graph_from_polygon(polygon=boundary_geom, network_type=network_type)

### Get details about services and location
- gets positions of hospitals 
- gets positions of fire stations 
- gets boundaries of administrative districts i.e. parts of a city that can be used for analysis 

In [ ]:
tags_hospitals = {"amenity": ["hospital"]}
hospitals = ox.features_from_polygon(polygon=boundary_geom, tags=tags_hospitals)

tags_fire_stations = {"amenity": ["fire_station"]}
fire_stations = ox.features_from_polygon(polygon=boundary_geom, tags=tags_fire_stations)

tags_administrative_districts = {"boundary": ["administrative"]}
administrative_districts = ox.features_from_polygon(polygon=boundary_geom, tags=tags_administrative_districts)

### Persist the data for usage in the next steps

In [ ]:
# persist those graphs
ox.save_graphml(road_network, filepath=cache_dir / f"network/drive_graph_{place_name}.graphml")

In [ ]:
hospitals.to_file(cache_dir / f"services/hospitals_{place_name}.geojson", driver="GeoJSON")
fire_stations.to_file(cache_dir / f"services/fire_stations_{place_name}.geojson", driver="GeoJSON")
administrative_districts.to_file(
    cache_dir / f"services/administrative_districts_{place_name}.geojson", driver="GeoJSON"
)